In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../data/preprocess/strong_zone/train.csv")
val = pd.read_csv("../data/preprocess/strong_zone/validation.csv")

print("Training:", train.shape)
print("Validation:", val.shape)

Training: (70525, 27)
Validation: (17571, 27)


In [3]:
print(train["label"].value_counts())

print(val["label"].value_counts())

label
0.0    49477
1.0    21048
Name: count, dtype: int64
label
0.0    12492
1.0     5079
Name: count, dtype: int64


In [4]:
X_train = train.drop(columns=["label"])
y_train = train["label"]

X_val = val.drop(columns=["label"])
y_val = val["label"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (70525, 26)
y_train: (70525,)


Dummy Classifier

In [5]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)

,strategy,'most_frequent'
,random_state,None
,constant,None


In [6]:
dummy_predictions = dummy_model.predict(X_val)

print(dummy_predictions[:10])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [7]:
from sklearn.metrics import accuracy_score, f1_score

dummy_accuracy = accuracy_score(y_val, dummy_predictions)
dummy_f1 = f1_score(y_val, dummy_predictions, average="macro")

print("Dummy Accuracy:", round(dummy_accuracy, 4))
print("Dummy Macro-F1:", round(dummy_f1, 4))

Dummy Accuracy: 0.7109
Dummy Macro-F1: 0.4155


In [8]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_val, dummy_predictions))

[[12492     0]
 [ 5079     0]]


In [9]:
print(classification_report(
    y_val,
    dummy_predictions,
    zero_division=0
))

              precision    recall  f1-score   support

         0.0       0.71      1.00      0.83     12492
         1.0       0.00      0.00      0.00      5079

    accuracy                           0.71     17571
   macro avg       0.36      0.50      0.42     17571
weighted avg       0.51      0.71      0.59     17571



Logistic Regression

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [11]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [12]:
logistic_predictions = logistic_model.predict(X_val)

In [13]:
from sklearn.metrics import accuracy_score, f1_score

print("Accuracy:", accuracy_score(y_val, logistic_predictions))

print("Macro-F1:", f1_score(
    y_val,
    logistic_predictions,
    average="macro"
))

Accuracy: 0.7099766660975471
Macro-F1: 0.4206593774490782


In [14]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_val, logistic_predictions))

print(classification_report(
    y_val,
    logistic_predictions,
    zero_division=0
))

[[12446    46]
 [ 5050    29]]
              precision    recall  f1-score   support

         0.0       0.71      1.00      0.83     12492
         1.0       0.39      0.01      0.01      5079

    accuracy                           0.71     17571
   macro avg       0.55      0.50      0.42     17571
weighted avg       0.62      0.71      0.59     17571



In [15]:
print(train["numeric__width_atr"].describe())

count    7.052500e+04
mean    -3.385216e-17
std      1.000007e+00
min     -8.840057e-01
25%     -5.897206e-01
50%     -2.951623e-01
75%      1.282167e-01
max      1.763242e+01
Name: numeric__width_atr, dtype: float64


Add weight and again logistic regression

In [16]:
balanced_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

balanced_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [17]:
balanced_predictions = balanced_model.predict(X_val)

In [18]:
print("Accuracy:",
      accuracy_score(y_val, balanced_predictions))

print("Macro-F1:",
      f1_score(y_val, balanced_predictions, average="macro"))

print(confusion_matrix(y_val, balanced_predictions))

print(classification_report(
    y_val,
    balanced_predictions,
    zero_division=0
))

Accuracy: 0.6089579420636275
Macro-F1: 0.5404554555839538
[[8742 3750]
 [3121 1958]]
              precision    recall  f1-score   support

         0.0       0.74      0.70      0.72     12492
         1.0       0.34      0.39      0.36      5079

    accuracy                           0.61     17571
   macro avg       0.54      0.54      0.54     17571
weighted avg       0.62      0.61      0.62     17571



Random forest

In [19]:
from sklearn.ensemble import RandomForestClassifier

In [20]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
rf_predictions = rf_model.predict(X_val)

In [22]:
print("Random Forest Accuracy:",
      accuracy_score(y_val, rf_predictions))

print("Random Forest Macro-F1:",
      f1_score(y_val, rf_predictions, average="macro"))

Random Forest Accuracy: 0.703659438848102
Random Forest Macro-F1: 0.4497436293296503


In [23]:
print(confusion_matrix(y_val, rf_predictions))

print(classification_report(
    y_val,
    rf_predictions,
    zero_division=0
))

[[12150   342]
 [ 4865   214]]
              precision    recall  f1-score   support

         0.0       0.71      0.97      0.82     12492
         1.0       0.38      0.04      0.08      5079

    accuracy                           0.70     17571
   macro avg       0.55      0.51      0.45     17571
weighted avg       0.62      0.70      0.61     17571



Balanced Random forest

In [24]:
rf_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_balanced.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
rf_balanced_predictions = rf_balanced.predict(X_val)

print("Balanced RF Accuracy:",
      accuracy_score(y_val, rf_balanced_predictions))

print("Balanced RF Macro-F1:",
      f1_score(y_val, rf_balanced_predictions, average="macro"))

print(confusion_matrix(y_val, rf_balanced_predictions))

print(classification_report(
    y_val,
    rf_balanced_predictions,
    zero_division=0
))

Balanced RF Accuracy: 0.7067895964942235
Balanced RF Macro-F1: 0.44506129779965065
[[12243   249]
 [ 4903   176]]
              precision    recall  f1-score   support

         0.0       0.71      0.98      0.83     12492
         1.0       0.41      0.03      0.06      5079

    accuracy                           0.71     17571
   macro avg       0.56      0.51      0.45     17571
weighted avg       0.63      0.71      0.61     17571



threshold  = 0.4

In [26]:
rf_probabilities = rf_model.predict_proba(X_val)[:, 1]

In [27]:
rf_predictions_04 = (rf_probabilities >= 0.4).astype(int)

In [28]:
print(confusion_matrix(y_val, rf_predictions_04))

print(f1_score(y_val, rf_predictions_04, average="macro"))

[[10550  1942]
 [ 4065  1014]]
0.5153962463030821


check different thresholds 

In [29]:
for threshold in [0.3, 0.35, 0.4, 0.45, 0.5]:

    predictions = (rf_probabilities >= threshold).astype(int)

    score = f1_score(y_val, predictions, average="macro")

    print(threshold, round(score, 4))

0.3 0.4947
0.35 0.526
0.4 0.5154
0.45 0.485
0.5 0.4553


In [30]:
rf_predictions_035 = (rf_probabilities >= 0.35).astype(int)

print(confusion_matrix(y_val, rf_predictions_035))

print(classification_report(
    y_val,
    rf_predictions_035,
    zero_division=0
))

[[8768 3724]
 [3290 1789]]
              precision    recall  f1-score   support

         0.0       0.73      0.70      0.71     12492
         1.0       0.32      0.35      0.34      5079

    accuracy                           0.60     17571
   macro avg       0.53      0.53      0.53     17571
weighted avg       0.61      0.60      0.61     17571



XGBOOST

In [31]:
from xgboost import XGBClassifier

ModuleNotFoundError: No module named 'xgboost'

In [34]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 1.1 MB/s eta 0:00:43
    --------------------------------------- 0.8/48.9 MB 1.1 MB/s eta 0:00:43
    --------------------------------------- 1.0/48.9 MB 1.1 MB/s eta 0:00:42
   - -------------------------------------- 1.3/48.9 MB 1.2 MB/s eta 0:00:41
   - -------------------------------------- 1.8/48.9 MB 1.3 MB/s eta 0:00:37
   - -------------------------------------- 2.1/48.9 MB 1.3 MB/s eta 0:00:36
   -- ------------------------------------- 2.6/48.9 MB 1.4 MB/s eta 0:00:34
   -- ------------------------------------- 2.9/48.9 MB 1.5 MB/s eta 0:00:32
   -- ------------------------------------- 3.4/48.9 MB 1.5 MB/s eta 0:00:30
   --- ------------


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
from xgboost import XGBClassifier

In [36]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [37]:
xgb_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [38]:
xgb_predictions = xgb_model.predict(X_val)

In [39]:
print("XGBoost Accuracy:",
      accuracy_score(y_val, xgb_predictions))

print("XGBoost Macro-F1:",
      f1_score(y_val, xgb_predictions, average="macro"))

XGBoost Accuracy: 0.7103750497979625
XGBoost Macro-F1: 0.42136219725355406


In [40]:
print(confusion_matrix(y_val, xgb_predictions))

print(classification_report(
    y_val,
    xgb_predictions,
    zero_division=0
))

[[12450    42]
 [ 5047    32]]
              precision    recall  f1-score   support

         0.0       0.71      1.00      0.83     12492
         1.0       0.43      0.01      0.01      5079

    accuracy                           0.71     17571
   macro avg       0.57      0.50      0.42     17571
weighted avg       0.63      0.71      0.59     17571



In [41]:
xgb_probabilities = xgb_model.predict_proba(X_val)[:, 1]

print(xgb_probabilities[:10])

[0.29317656 0.4270744  0.29500666 0.2365626  0.31038338 0.30755436
 0.27177945 0.28152236 0.32866082 0.31862077]


Balanced XGBoost

In [42]:
weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Class weight:", weight)

Class weight: 2.350674648422653


In [43]:
xgb_balanced = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_balanced.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [44]:
xgb_balanced_predictions = xgb_balanced.predict(X_val)

print("Accuracy:",
      accuracy_score(y_val, xgb_balanced_predictions))

print("Macro-F1:",
      f1_score(y_val, xgb_balanced_predictions, average="macro"))

print(confusion_matrix(y_val, xgb_balanced_predictions))

print(classification_report(
    y_val,
    xgb_balanced_predictions,
    zero_division=0
))

Accuracy: 0.5754937112287292
Macro-F1: 0.5297909916959995
[[7795 4697]
 [2762 2317]]
              precision    recall  f1-score   support

         0.0       0.74      0.62      0.68     12492
         1.0       0.33      0.46      0.38      5079

    accuracy                           0.58     17571
   macro avg       0.53      0.54      0.53     17571
weighted avg       0.62      0.58      0.59     17571



Balanced XGBoost threshold tuning

In [45]:
xgb_probs = xgb_balanced.predict_proba(X_val)[:, 1]

In [46]:
thresholds = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]

for t in thresholds:
    predictions = (xgb_probs >= t).astype(int)

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(y_val, predictions, average="macro")

    print(t, round(accuracy, 4), round(macro_f1, 4))

0.35 0.2922 0.2303
0.4 0.3091 0.2634
0.45 0.412 0.4102
0.5 0.5755 0.5298
0.55 0.6676 0.5259
0.6 0.6997 0.4793
0.65 0.7092 0.4365


In [47]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("ROC-AUC:", roc_auc_score(y_val, xgb_probs))

print("PR-AUC:", average_precision_score(y_val, xgb_probs))

ROC-AUC: 0.5586915653582775
PR-AUC: 0.338669077256611


check overfitting

In [48]:
train_predictions = xgb_balanced.predict(X_train)
val_predictions = xgb_balanced.predict(X_val)

print("Train Macro-F1:",
      f1_score(y_train, train_predictions, average="macro"))

print("Validation Macro-F1:",
      f1_score(y_val, val_predictions, average="macro"))

Train Macro-F1: 0.5613116599828236
Validation Macro-F1: 0.5297909916959995


In [49]:
importance = pd.Series(
    xgb_balanced.feature_importances_,
    index=X_train.columns
)

print(importance.sort_values(ascending=False).head(15))

categorical__origin_type_displacement_body            0.140816
numeric__width_atr                                    0.122572
categorical__direction_bearish                        0.085992
categorical__origin_type_bullish_rejection_cluster    0.042427
categorical__source_timeframe_4h                      0.041208
categorical__zone_family_reaction                     0.041206
numeric__signed_distance_atr                          0.039248
numeric__normalized_atr                               0.038869
numeric__rolling_mean_return_20                       0.037847
categorical__zone_family_imbalance                    0.035304
categorical__source_timeframe_15m                     0.032873
numeric__overlapping_candidate_count_at_decision      0.030047
numeric__relative_volume_20                           0.030035
numeric__close_location_value                         0.029913
categorical__origin_type_bullish_fvg                  0.029573
dtype: float32


feature ablation experiment

In [50]:
overlap_columns = [
    "numeric__overlapping_family_count_at_decision",
    "numeric__overlapping_candidate_count_at_decision"
]

X_train_no_overlap = X_train.drop(columns=overlap_columns)
X_val_no_overlap = X_val.drop(columns=overlap_columns)

print(X_train_no_overlap.shape)
print(X_val_no_overlap.shape)

(70525, 24)
(17571, 24)


In [51]:
xgb_no_overlap = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_no_overlap.fit(X_train_no_overlap, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [52]:
no_overlap_predictions = xgb_no_overlap.predict(X_val_no_overlap)

print("With overlap:", 0.5298)

print("Without overlap:",
      f1_score(y_val, no_overlap_predictions, average="macro"))

With overlap: 0.5298
Without overlap: 0.5336145784091795


XGBoost Hyperparameter Tuning
tree depth


In [53]:
for depth in [2, 3, 4, 5, 6]:

    model = XGBClassifier(
        n_estimators=200,
        max_depth=depth,
        learning_rate=0.05,
        scale_pos_weight=weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    score = f1_score(y_val, predictions, average="macro")

    print("Depth:", depth, "Macro-F1:", round(score, 4))

Depth: 2 Macro-F1: 0.5346
Depth: 3 Macro-F1: 0.5323
Depth: 4 Macro-F1: 0.5298
Depth: 5 Macro-F1: 0.5282
Depth: 6 Macro-F1: 0.5271


In [54]:
xgb_depth2_no_overlap = XGBClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_depth2_no_overlap.fit(X_train_no_overlap, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [55]:
predictions = xgb_depth2_no_overlap.predict(X_val_no_overlap)

print("Macro-F1:", f1_score(y_val, predictions, average="macro"))

print("Accuracy:", accuracy_score(y_val, predictions))

print(confusion_matrix(y_val, predictions))

Macro-F1: 0.5342956685941352
Accuracy: 0.5822093221785897
[[7933 4559]
 [2782 2297]]


Depth 2 model

In [56]:
xgb_depth2 = XGBClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_depth2.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [57]:
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import roc_auc_score, average_precision_score

models = {
    "Balanced Logistic": balanced_model,
    "XGBoost Depth 2": xgb_depth2
}

for name, model in models.items():
    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)[:, 1]

    print("\n", name)
    print("Accuracy:", round(accuracy_score(y_val, predictions), 4))
    print("Macro-F1:", round(f1_score(y_val, predictions, average="macro"), 4))
    print("Strong Precision:", round(precision_score(y_val, predictions), 4))
    print("Strong Recall:", round(recall_score(y_val, predictions), 4))
    print("ROC-AUC:", round(roc_auc_score(y_val, probabilities), 4))
    print("PR-AUC:", round(average_precision_score(y_val, probabilities), 4))


 Balanced Logistic
Accuracy: 0.609
Macro-F1: 0.5405
Strong Precision: 0.343
Strong Recall: 0.3855
ROC-AUC: 0.562
PR-AUC: 0.3414

 XGBoost Depth 2
Accuracy: 0.582
Macro-F1: 0.5346
Strong Precision: 0.3355
Strong Recall: 0.455
ROC-AUC: 0.5638
PR-AUC: 0.3436
